# dNATY — Business Benchmark (GPU T4)

Testa o dNATY em **3 datasets reais de negócio**:
- **Bank Marketing** (UCI) — prever se cliente assina depósito a prazo
- **Adult Income** (UCI) — prever renda > $50K
- **Credit Default** (UCI) — prever inadimplência em cartão de crédito

**Runtime → Change runtime type → T4 GPU** antes de rodar.

In [ ]:
# ── SETUP ─────────────────────────────────────────────────────────────────────
import subprocess, sys, os, shutil
import torch

assert torch.cuda.is_available(), 'Sem GPU — vai em Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory//1024**3}GB')

# Clona o repo
REPO_URL = 'https://github.com/pedrovergueiro/dnaty-saas.git'
if not os.path.exists('/content/dnaty-saas'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/dnaty-saas'], check=True)
else:
    subprocess.run(['git', '-C', '/content/dnaty-saas', 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', '/content/dnaty-saas', 'reset', '--hard', 'origin/main'], check=True)

for root, dirs, _ in os.walk('/content/dnaty-saas'):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

if '/content/dnaty-saas' not in sys.path:
    sys.path.insert(0, '/content/dnaty-saas')

subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn', 'pandas', '-q'], check=True)
print('Setup OK')

In [ ]:
# ── HELPERS ───────────────────────────────────────────────────────────────────
import time
import numpy as np
import pandas as pd
import torch
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from dnaty.evolution.evolver import DnatyEvolver

DEVICE = 'cuda'

def preprocess(df, target_col):
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)
    y = df[target_col].values.astype(int)
    X = df.drop(columns=[target_col]).values.astype(np.float32)
    X = StandardScaler().fit_transform(X)
    return X, y

def make_loaders(X, y, batch_size=1024):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    def loader(Xd, yd, shuffle):
        ds = TensorDataset(torch.tensor(Xd), torch.tensor(yd, dtype=torch.long))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=True)
    return loader(X_tr, y_tr, True), loader(X_te, y_te, False), X_tr.shape[1], len(np.unique(y))

def run_dnaty(name, train_loader, test_loader, input_size, n_classes, n_pop=15, n_generations=20):
    print(f'\n{"="*60}')
    print(f'dNATY x {name}')
    print(f'Input: {input_size} features | Classes: {n_classes} | GPU: {torch.cuda.get_device_name(0)}')
    print(f'{"="*60}')

    evolver = DnatyEvolver(
        n_pop=n_pop, n_generations=n_generations,
        t_local=3, lr=1e-3, lambda1=1e-4, lambda2=1e-3,
        device=DEVICE, input_size=input_size, n_classes=n_classes,
        init_hidden=[256, 128], batch_size=1024, verbose=True,
    )

    t0 = time.time()
    best, history = evolver.run(train_loader, test_loader)
    duration = time.time() - t0

    print(f'\n--- RESULTADO: {name} ---')
    print(f'Acuracia:    {best.acc*100:.2f}%')
    print(f'Parametros:  {best.count_params():,}')
    print(f'Tempo:       {duration:.1f}s ({duration/len(history):.1f}s/gen)')
    print(f'Arquitetura: {best.model.layer_sizes}')
    print(f'Ativacoes:   {best.model.activations}')
    print(f'Progressao:')
    for log in history:
        bar = '#' * int(log.best_acc * 30)
        print(f'  Gen {log.gen:02d}: {log.best_acc*100:.2f}% |{bar}|')
    return best.acc, duration, len(history), best.model.layer_sizes

print('Helpers prontos')

In [ ]:
# ── DATASET 1: Bank Marketing ─────────────────────────────────────────────────
# 45.211 clientes de banco | prever se assina deposito a prazo
print('Baixando Bank Marketing (OpenML #1461)...')
data = fetch_openml(data_id=1461, as_frame=True, parser='auto')
df = data.frame.copy()
# target e 'Class' com valores '1' e '2'
df['y'] = (df['Class'].astype(str) == '2').astype(int)
df = df.drop(columns=['Class'])
print(f'Linhas: {len(df):,} | Positivos (assinou): {df.y.sum():,} ({df.y.mean()*100:.1f}%)')

X1, y1 = preprocess(df, 'y')
train1, test1, inp1, nc1 = make_loaders(X1, y1)
acc1, dur1, g1, arch1 = run_dnaty('Bank Marketing — Conversao de clientes', train1, test1, inp1, nc1)

In [ ]:
# ── DATASET 2: Adult Income ───────────────────────────────────────────────────
# 48.842 pessoas | prever renda > $50K (usado em credito e seguros)
print('Baixando Adult Income (OpenML #1590)...')
data2 = fetch_openml(data_id=1590, as_frame=True, parser='auto')
df2 = data2.frame.copy()
df2['y'] = (df2['class'].astype(str).str.strip() == '>50K').astype(int)
df2 = df2.drop(columns=['class'])
print(f'Linhas: {len(df2):,} | Renda alta: {df2.y.sum():,} ({df2.y.mean()*100:.1f}%)')

X2, y2 = preprocess(df2, 'y')
train2, test2, inp2, nc2 = make_loaders(X2, y2)
acc2, dur2, g2, arch2 = run_dnaty('Adult Income — Predicao de renda', train2, test2, inp2, nc2)

In [ ]:
# ── DATASET 3: Credit Default (Taiwan) ────────────────────────────────────────
# 30.000 clientes de cartao de credito | prever inadimplencia
print('Baixando Credit Card Default (OpenML #42477)...')
data3 = fetch_openml(data_id=42477, as_frame=True, parser='auto')
df3 = data3.frame.copy()
target_col = [c for c in df3.columns if 'default' in c.lower() or 'pay' in c.lower()][-1]
df3['y'] = df3[target_col].astype(str).map(lambda x: 1 if x.strip() in ['1','yes','Yes','true','True'] else 0)
df3 = df3.drop(columns=[target_col])
print(f'Linhas: {len(df3):,} | Inadimplentes: {df3.y.sum():,} ({df3.y.mean()*100:.1f}%)')

X3, y3 = preprocess(df3, 'y')
train3, test3, inp3, nc3 = make_loaders(X3, y3)
acc3, dur3, g3, arch3 = run_dnaty('Credit Default — Risco de inadimplencia', train3, test3, inp3, nc3)

In [ ]:
# ── RESUMO FINAL ──────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('RESUMO — dNATY em Dados Reais de Negócio (GPU T4)')
print('='*60)
print(f'{"Dataset":<35} {"Acuracia":>10} {"Tempo":>8} {"Gens":>6}')
print('-'*60)
print(f'{"Bank Marketing (41k clientes)":<35} {acc1*100:>9.2f}% {dur1:>7.1f}s {g1:>6}')
print(f'{"Adult Income (48k pessoas)":<35} {acc2*100:>9.2f}% {dur2:>7.1f}s {g2:>6}')
print(f'{"Credit Default (30k cartoes)":<35} {acc3*100:>9.2f}% {dur3:>7.1f}s {g3:>6}')
print('='*60)
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Tempo total: {(dur1+dur2+dur3)/60:.1f} minutos')
print('\ndNATY encontrou a melhor arquitetura para cada problema')
print('automaticamente — sem ajuste manual de hiperparametros.')